# Studi Kasus Decision Tree: Prediksi Status Pompa pada Sistem Irigasi Sawah

**Tujuan.** Notebook ini mendemonstrasikan mekanisme decision tree secara bertahap, mulai dari perhitungan manual entropy dan information gain hingga training tree dengan scikit-learn, pada satu kasus yang dapat direproduksi tanpa file eksternal maupun akun.

**Reproducibility.** Dibutuhkan Python 3.9+ dengan `numpy`, `pandas`, `scikit-learn`, dan `matplotlib`. Seluruh data dibangkitkan di dalam notebook dengan seed tetap. Jalankan semua cell secara berurutan (*Run All*).

**Struktur**

| Bagian | Isi | Data |
|---|---|---|
| A | Perhitungan manual entropy dan information gain, diverifikasi terhadap scikit-learn | Tabel mini 16 row, ditulis langsung di cell |
| B | Decision tree pada data yang lebih realistis: data generation, compression ke event, training, evaluation, pruning | Data sintetis dari simulator sistem irigasi |
| C | Keterbatasan dan pengembangan lanjutan | |

**Disclosure dan batasan data.**
1. Data pada Bagian B bersifat **sintetis**, dibangkitkan dari rule controller yang ditetapkan oleh penulis, dengan penambahan sensor noise dan intervensi manual acak. Tree yang di-training pada data ini pada dasarnya merekonstruksi rule tersebut, sehingga hasilnya tidak dapat dibaca sebagai temuan baru tentang kondisi sawah. Kontribusi studi ini terletak pada demonstrasi bagaimana tree merekonstruksi rule tersebut dan seberapa robust hasilnya terhadap noise.
2. Parameter fisik simulator (laju evaporasi, kenaikan kelembaban saat pompa ON, dan seterusnya) bernilai ilustratif dan **belum dikalibrasi** terhadap data lapangan. Klaim bahwa data "dimodelkan dari data lapangan" baru valid setelah parameter pada `PARAMS` diestimasi dari data nyata.
3. Seluruh angka pada output dihasilkan saat notebook dijalankan dengan seed tetap. Hasil dapat bervariasi tipis antar versi numpy dan scikit-learn.

In [ ]:
# Make sure dependencies are available: any missing package is installed via pip.
import importlib.util
import subprocess
import sys

PIP_PACKAGES = {
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "sklearn": "scikit-learn",
}

missing = [package for module, package in PIP_PACKAGES.items() if importlib.util.find_spec(module) is None]
if missing:
    print("Installing missing packages:", ", ".join(missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
    print("Done. If the import in the next cell still fails, restart the kernel and run the cells again.")
else:
    print("All dependencies are already installed.")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.metrics import balanced_accuracy_score

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 20)
plt.rcParams.update({"figure.dpi": 100, "axes.grid": True, "grid.alpha": 0.3})
SEED = 42

---
# Bagian A. Perhitungan Manual pada Dataset Mini

Dataset mini terdiri atas 16 observasi dengan tiga feature categorical dan satu target binary.

| Feature | Value |
|---|---|
| `soil` | dry, moist, wet |
| `rain` | yes, no |
| `water` (tinggi air di tangki) | low, enough |
| **Target** `pump` | ON, OFF |

Rule pembangkit data: pompa ON jika `soil = dry`, `water = enough`, dan `rain = no`. Row nomor 14 sengaja melanggar rule tersebut untuk merepresentasikan intervensi manual petani (label noise). Dengan demikian tree tidak dapat mencapai akurasi sempurna, dan terdapat residual error yang dapat dianalisis.

In [ ]:
mini = pd.DataFrame([
    # soil,   rain,   water,    pump
    ("dry",   "no",   "enough", "ON"),
    ("dry",   "no",   "enough", "ON"),
    ("dry",   "no",   "enough", "ON"),
    ("dry",   "no",   "low",    "OFF"),
    ("dry",   "yes",  "enough", "OFF"),
    ("dry",   "no",   "enough", "ON"),
    ("moist", "no",   "enough", "OFF"),
    ("moist", "no",   "enough", "OFF"),
    ("moist", "yes",  "enough", "OFF"),
    ("moist", "no",   "low",    "OFF"),
    ("wet",   "no",   "enough", "OFF"),
    ("wet",   "yes",  "enough", "OFF"),
    ("dry",   "yes",  "low",    "OFF"),
    ("moist", "no",   "enough", "ON"),    # manual intervention (rule violator)
    ("wet",   "no",   "enough", "OFF"),
    ("dry",   "no",   "enough", "ON"),
], columns=["soil", "rain", "water", "pump"])
mini.index = mini.index + 1
mini

## A1. Entropy dan Gini

Untuk dataset dengan proportion class $p_i$:

$$H = -\sum_i p_i \log_2 p_i \qquad\qquad G = 1 - \sum_i p_i^2$$

Nilai 0 menandakan node yang pure (hanya satu class). Pada kasus dua class, entropy maksimum (1 bit) dicapai ketika proportion kedua class sama, yaitu 50:50.

**Information gain** suatu split didefinisikan sebagai selisih entropy sebelum split dan rata-rata entropy sesudah split, yang di-weight dengan proporsi jumlah row pada tiap branch:

$$IG = H(\text{parent}) - \sum_k \frac{n_k}{n}\, H(\text{child}_k)$$

In [ ]:
def entropy(y):
    p = y.value_counts(normalize=True)
    return float(-(p * np.log2(p)).sum())

def gini(y):
    p = y.value_counts(normalize=True)
    return float(1 - (p ** 2).sum())

def binary_information_gain(df, column, value, target="pump"):
    # Binary split: (column == value) versus the rest, which is what CART/scikit-learn does.
    yes = df[df[column] == value]
    no = df[df[column] != value]
    entropy_after = len(yes) / len(df) * entropy(yes[target]) + len(no) / len(df) * entropy(no[target])
    return entropy(df[target]) - entropy_after

n_on = (mini.pump == "ON").sum()
print(f"Rows         : {len(mini)}  (ON = {n_on}, OFF = {len(mini) - n_on})")
print(f"Root entropy : {entropy(mini.pump):.4f} bit")
print(f"Root gini    : {gini(mini.pump):.4f}")

## A2. Pemilihan split pada root

scikit-learn (CART) hanya menghasilkan split **binary**. Feature dengan tiga value (`soil`) karenanya diuraikan menjadi tiga candidate question yes/no (misalnya `soil = dry?`). Information gain dihitung untuk seluruh candidate, kemudian diurutkan secara menurun. Candidate dengan information gain tertinggi dipilih sebagai split pada root.

In [ ]:
candidates = []
for column in ["soil", "rain", "water"]:
    for value in mini[column].unique():
        candidates.append({"question": f"{column} = {value}?",
                           "information_gain": binary_information_gain(mini, column, value)})

ig_table = (pd.DataFrame(candidates)
            .sort_values("information_gain", ascending=False)
            .reset_index(drop=True))
ig_table["information_gain"] = ig_table["information_gain"].round(4)
ig_table

In [ ]:
best_split = ig_table.iloc[0]["question"]
column, value = best_split.replace("?", "").split(" = ")
print(f"Best root split: {best_split}\n")

branch_detail = []
for branch, subset in [("YES", mini[mini[column] == value]), ("NO", mini[mini[column] != value])]:
    branch_detail.append({"branch": branch, "n": len(subset),
                          "ON": (subset.pump == "ON").sum(), "OFF": (subset.pump == "OFF").sum(),
                          "entropy": round(entropy(subset.pump), 4)})
branch_detail = pd.DataFrame(branch_detail)
display(branch_detail)

entropy_after = sum(r.n / len(mini) * r.entropy for r in branch_detail.itertuples())
print(f"\nEntropy before  : {entropy(mini.pump):.4f}")
print(f"Entropy after   : {entropy_after:.4f}  (weighted average)")
print(f"Information gain: {entropy(mini.pump) - entropy_after:.4f}")

## A3. Split pada level kedua

Branch `NO` (`soil` bukan `dry`) hampir seluruhnya OFF, dengan satu row pelanggar. Branch `YES` masih campuran (impure), sehingga prosedur yang sama diterapkan secara rekursif pada subset row di branch tersebut.

In [ ]:
subset = mini[mini[column] == value]
print(f"Subset: {column} = {value}  ({len(subset)} rows, entropy {entropy(subset.pump):.4f})\n")

candidates2 = []
for feature in ["rain", "water"]:
    for feature_value in subset[feature].unique():
        candidates2.append({"question": f"{feature} = {feature_value}?",
                            "information_gain": round(binary_information_gain(subset, feature, feature_value), 4)})
display(pd.DataFrame(candidates2).sort_values("information_gain", ascending=False).reset_index(drop=True))

## A4. Verifikasi terhadap scikit-learn

Jika perhitungan manual benar, tree scikit-learn dengan `criterion="entropy"` harus memilih split root yang sama dengan nilai information gain yang identik. Kesesuaian tersebut diverifikasi secara eksplisit melalui `assert`.

In [ ]:
X_mini = pd.get_dummies(mini[["soil", "rain", "water"]]).astype(int)
y_mini = mini["pump"]

tree_mini = DecisionTreeClassifier(criterion="entropy", random_state=0).fit(X_mini, y_mini)
t = tree_mini.tree_

# Root information gain according to scikit-learn, computed from the impurity of each node.
left, right = t.children_left[0], t.children_right[0]
ig_sklearn = t.impurity[0] - (t.n_node_samples[left] / t.n_node_samples[0] * t.impurity[left]
                              + t.n_node_samples[right] / t.n_node_samples[0] * t.impurity[right])

print("Root split (scikit-learn)       :", X_mini.columns[t.feature[0]])
print(f"Root information gain           : {ig_sklearn:.4f}")
print(f"Hand-computed information gain  : {ig_table.iloc[0]['information_gain']:.4f}")
assert abs(ig_sklearn - ig_table.iloc[0]["information_gain"]) < 1e-3, "hand computation does not match"
print("\nThey match.\n")

print(export_text(tree_mini, feature_names=list(X_mini.columns)))

In [ ]:
from matplotlib.patches import FancyBboxPatch
from matplotlib.colors import to_rgb

def plot_clean_tree(clf, feature_names, title, unit="time"):
    """Draw the tree top-down: one-line conditions, class-coloured leaves, no filler."""
    t = clf.tree_
    classes = list(clf.classes_)
    palette = {"ON": "#f4a261", "OFF": "#90caf9"}

    # Depth becomes the x axis; leaves are stacked on the y axis in order.
    depth_of, y_of = {}, {}
    next_leaf = [0]
    def place(node, depth):
        left, right = t.children_left[node], t.children_right[node]
        if left == right:
            depth_of[node], y_of[node] = depth, -next_leaf[0]
            next_leaf[0] += 1
        else:
            place(left, depth + 1)
            place(right, depth + 1)
            y_of[node] = (y_of[left] + y_of[right]) / 2
            depth_of[node] = depth
    place(0, 0)

    n_leaves = max(next_leaf[0], 1)
    max_depth = max(depth_of.values())
    step_x, box_w, box_h = 2.3, 1.5, 0.52
    fig, ax = plt.subplots(figsize=(step_x * (max_depth + 1) + 0.8, 0.72 * n_leaves + 1.0))

    def node_text(node):
        left, right = t.children_left[node], t.children_right[node]
        if left == right:                          # leaf: predicted class + share of the data
            value = t.value[node][0] / t.value[node][0].sum()
            class_label = classes[int(value.argmax())]
            share = 100 * t.weighted_n_node_samples[node] / t.weighted_n_node_samples[0]
            return f"{class_label}\n{share:.0f}% of {unit}", class_label, value.max()
        feature, value = feature_names[t.feature[node]].split("_", 1)   # split: one yes/no question
        return f"{feature} = {value}?", None, None

    for node in range(t.node_count):
        text, class_label, purity = node_text(node)
        x, y = depth_of[node] * step_x, y_of[node]
        left, right = t.children_left[node], t.children_right[node]
        x_right = x + box_w / 2

        if left != right:
            for child, answer in [(left, "no"), (right, "yes")]:
                x_child = depth_of[child] * step_x - box_w / 2
                ax.plot([x_right, x_child], [y, y_of[child]], color="#bdbdbd", lw=1.1, zorder=1)
                ax.text(x_right + (x_child - x_right) * 0.55, y + (y_of[child] - y) * 0.55, answer,
                        ha="center", va="center", fontsize=8, color="#616161",
                        bbox=dict(facecolor="white", edgecolor="none", pad=0.3), zorder=3)

        # Leaf colour = predicted class, stronger when the leaf is purer.
        face = "#f5f5f5" if class_label is None else tuple(1 - purity * (1 - np.array(to_rgb(palette[class_label]))))
        ax.add_patch(FancyBboxPatch((x - box_w / 2, y - box_h / 2), box_w, box_h,
                                    boxstyle="round,pad=0.02,rounding_size=0.1",
                                    facecolor=face, edgecolor="#9e9e9e", lw=0.9, zorder=2))
        ax.text(x, y, text, ha="center", va="center", fontsize=10, color="#212121",
                linespacing=1.35, zorder=4)

    ax.set_xlim(-box_w / 2 - 0.2, max_depth * step_x + box_w / 2 + 0.2)
    ax.set_ylim(min(y_of.values()) - 0.55, max(y_of.values()) + 0.55)
    ax.axis("off")
    ax.set_title(title, fontsize=12, pad=10)
    plt.tight_layout()
    plt.show()

plot_clean_tree(tree_mini, list(X_mini.columns), "Mini dataset tree", unit="rows")

**Catatan interpretasi.**
- Row 14 memiliki value feature identik dengan row 7 dan 8 (`soil = moist`, `rain = no`, `water = enough`) tetapi label berbeda. Tree **tidak dapat** memisahkannya dengan feature yang tersedia, sehingga leaf tersebut tetap campuran dan diprediksi sebagai class mayoritas (OFF). Ini merupakan contoh *label noise*, yaitu irreducible error yang tidak dapat dihilangkan oleh tree sedalam apa pun.
- Beberapa branch pada tree tanpa batas depth berakhir pada leaf dengan prediction yang sama (OFF). Hal ini terjadi karena algoritma tetap memecah node yang belum pure walaupun prediction tidak berubah. Pada Bagian B, depth dibatasi untuk menghindari split yang redundan seperti ini.
- Pada keluaran `export_text`, kondisi `soil_dry <= 0.50` berarti jawaban **No** (value 0), sedangkan `> 0.50` berarti **Yes** (value 1).
- Algoritma ID3/C4.5 menggunakan split *multiway* (satu branch per value), sedangkan CART pada scikit-learn menggunakan split binary. Akibatnya, nilai information gain dapat berbeda dari literatur yang berbasis ID3.

---
# Bagian B. Decision Tree pada Data Sintetis yang Lebih Realistis

Dataset mini pada Bagian A terlalu bersih untuk merepresentasikan data sensor. Pada Bagian B dibangkitkan data sensor sintetis selama 90 hari (nilai default `days`) dengan resolusi 1 menit, yang kemudian diperlakukan seperti data lapangan: di-binning menjadi category, dikompresi menjadi event, dan di-evaluate dengan split temporal.

## B1. Parameter simulator

Seluruh parameter dikumpulkan pada satu dictionary (`PARAMS`) agar mudah dimodifikasi. **Value yang digunakan bersifat ilustratif.** Preset dipilih agar sistem berada pada regime kerja yang wajar (pompa ON sekitar seperempat waktu, tanah tidak kering kronis, tangki tidak habis terus-menerus), sehingga tidak ada satu state yang mendominasi data. Komentar pada sisi kanan setiap parameter menunjukkan cara mengestimasinya dari data nyata.

In [ ]:
PARAMS = {
    # --- environment ---
    "days": 90,
    "initial_moisture": 45.0,          # % (relative water content)
    "initial_water_level": 12.0,       # arbitrary unit (e.g. dm) in the tank
    "daytime_evaporation": 0.08,       # %/min, 08:00-16:00  <- moisture slope when pump OFF, no rain, daytime
    "night_evaporation": 0.03,         # %/min, other hours    <- same, at night
    "pump_moisture_gain": 0.15,        # %/min while pump ON   <- moisture slope while pump ON
    "sensor_lag": 20,                  # minutes: the root-zone sensor "sees" water after it seeps in <- lag between pump ON and the rise in reading
    "rain_moisture_gain": 0.12,        # %/min while raining   <- moisture slope while raining (light rain: only partly offsets daytime evaporation)
    "rain_probability": 0.0010,        # chance of rain starting per minute (about 1.4 rain events per day)
    "rain_duration": (45, 150),        # minutes (min, max)
    "water_drop_idle": 0.0005,         # water level/min (seepage, evaporation)
    "water_drop_pump": 0.004,          # extra water level/min while pump ON
    "tank_refill_hour": 6,             # the farmer refills the tank at 06:00 if it is below the threshold
    "tank_refill_level": 15.0,
    "sensor_noise": 0.4,               # std of moisture sensor noise (%)  <- std of successive differences on stable segments

    # --- pump control rules (the ones the tree should "discover") ---
    "turn_on_threshold": 40.0,         # pump turns on if the FILTERED reading < 40 %
    "minimum_water_level": 5.0,        # and water level >= 5
    "forbidden_hours": (12, 18),       # pump may not run 12:00-17:59 (avoid daytime evaporation)
    "min_on_minutes": 10,              # relay guard: once ON, it stays on at least 10 minutes
    "override_probability": 0.0005,    # chance of manual intervention per minute
    "override_duration": (10, 45),     # minutes

    # --- feature binning ---
    "dry_threshold": 40.0,             # soil: <=40 dry, <=50 moist, else wet. Equal to the pump threshold, so category and controller share one boundary
    "wet_threshold": 50.0,
    "debounce_window": 15,             # rolling median of 15 minutes, shared by the controller and the features (past data only, so it is usable for real time)
}
pd.Series({k: str(v) for k, v in PARAMS.items()}, name="value").to_frame()

## B2. Simulator

Pada setiap menit, simulator memperbarui kelembaban tanah dan tinggi air, kemudian controller menentukan status pompa dengan rule berikut:

$$\text{pump ON} \iff \text{filtered moisture} < 40\% \;\wedge\; \text{water level} \geq 5 \;\wedge\; \text{tidak hujan} \;\wedge\; \text{jam di luar 12:00-17:59}$$

Dua mekanisme non-ideal ditambahkan: `min_on_minutes` (pelindung relay, pompa bertahan ON minimal beberapa menit) dan **manual override** acak (`override_probability`) yang membalik keputusan controller selama 10-45 menit. Override inilah sumber *label noise* pada data.

Dari rule tersebut terdapat satu kondisi ON dan empat alasan OFF yang berbeda: (1) soil di atas threshold, (2) tangki rendah, (3) hujan, dan (4) jam larangan. Desain ini disengaja agar setiap leaf pada tree dapat ditelusuri ke satu penyebab.

Controller membaca *filtered reading*, yaitu median bergulir `debounce_window` menit dari pembacaan sensor mentah (noise `sensor_noise`, tertinggal `sensor_lag` menit). Filter yang sama dipakai untuk membentuk feature `soil`, dan batas category `dry` ditetapkan sama dengan threshold pompa. Dengan demikian tidak ada mismatch antara sinyal pengambil keputusan dan sinyal feature, sehingga ketidaksempurnaan tree hanya berasal dari override, `min_on_minutes`, dan keterbatasan depth. Konsekuensinya, data ini lebih bersih daripada kondisi lapangan (lihat Bagian C).

`sensor_lag` dipertahankan karena tanpa lag pembacaan segera melewati threshold setelah pompa menyala, sehingga lebih banyak menit ON jatuh pada category `moist` dan jumlah event membengkak akibat chattering di sekitar threshold (sensitivitas dapat diuji dengan `sensor_lag = 0`).

Empat feature digunakan: `soil`, `water`, `rain`, dan `time`. Bin `noon` (12-17) bertepatan dengan jam larangan, sedangkan bin `afternoon` (18-23) tidak dilarang.

In [ ]:
def simulate(params, seed=SEED):
    rng = np.random.default_rng(seed)
    n = params["days"] * 24 * 60
    timestamps = pd.date_range("2025-01-01", periods=n, freq="min")

    moisture, water_level = params["initial_moisture"], params["initial_water_level"]
    pump_on, rain_remaining, override_remaining, min_on_remaining = False, 0, 0, 0
    override_state = False
    moisture_history = []          # true moisture per minute, used to model the sensor lag
    reading_history = []           # raw sensor readings, used by the median filter
    rows = []

    for ts in timestamps:
        # weather
        if rain_remaining == 0 and rng.random() < params["rain_probability"]:
            rain_remaining = int(rng.integers(*params["rain_duration"]))
        raining = rain_remaining > 0
        rain_remaining = max(0, rain_remaining - 1)

        # physics
        evaporation = params["daytime_evaporation"] if 8 <= ts.hour < 16 else params["night_evaporation"]
        moisture = float(np.clip(moisture - evaporation + (params["rain_moisture_gain"] if raining else 0)
                                 + (params["pump_moisture_gain"] if pump_on else 0), 0, 100))
        water_level = float(np.clip(water_level - params["water_drop_idle"] - (params["water_drop_pump"] if pump_on else 0), 0, 15))
        if ts.hour == params["tank_refill_hour"] and ts.minute == 0 and water_level < params["minimum_water_level"]:
            water_level = params["tank_refill_level"]

        # sensor and pump decision
        moisture_history.append(moisture)
        sensor_moisture = moisture_history[max(0, len(moisture_history) - 1 - params["sensor_lag"])]   # sensor lags behind
        raw_reading = sensor_moisture + rng.normal(0, params["sensor_noise"])
        reading_history.append(raw_reading)
        filtered_reading = float(np.median(reading_history[-params["debounce_window"]:]))              # same causal median filter as the features
        time_allowed = not (params["forbidden_hours"][0] <= ts.hour < params["forbidden_hours"][1])
        automatic_on = (filtered_reading < params["turn_on_threshold"]) and (water_level >= params["minimum_water_level"]) and (not raining) and time_allowed

        if override_remaining == 0 and rng.random() < params["override_probability"]:
            override_remaining = int(rng.integers(*params["override_duration"]))
            override_state = not automatic_on
        if override_remaining > 0:
            pump_on, override_remaining = override_state, override_remaining - 1
        elif min_on_remaining > 0:
            pump_on, min_on_remaining = True, min_on_remaining - 1
        else:
            pump_on = automatic_on
            if automatic_on:
                min_on_remaining = params["min_on_minutes"] - 1

        rows.append((ts, raw_reading, water_level, raining, pump_on))

    return pd.DataFrame(rows, columns=["ts", "raw_moisture", "water_level", "rain", "pump"])

raw = simulate(PARAMS)
print(f"{len(raw):,} rows (1 row = 1 minute, {PARAMS['days']} days)")
raw.head()

### Visualisasi data simulasi

Plot menampilkan tiga hari data: garis biru adalah kelembaban (median filter 15 menit), area oranye adalah periode pompa ON, area biru muda adalah hujan, dan garis putus-putus pada panel atas adalah threshold 40%. Empat pola dapat diamati:
1. **Siklus ON/OFF di sekitar threshold.** Pada jam yang diizinkan, kelembaban berosilasi di sekitar 40%: pompa menyala begitu filtered reading turun di bawah threshold dan mati setelah pembacaan naik kembali.
2. **Jam larangan.** Pada 12:00-17:59 kelembaban turun jauh di bawah threshold (pada plot hingga sekitar 17-22%) karena pompa dilarang, dan pompa baru menyala lama setelah pukul 18:00.
3. **Tangki rendah.** Ketika tinggi air berada di bawah garis putus-putus pada panel bawah, pompa tidak menyala walaupun tanah dry, sehingga kelembaban terus turun sampai tangki diisi pukul 06:00.
4. **Hujan.** Pompa tidak menyala selama hujan, walaupun kelembaban berada di bawah threshold.

In [ ]:
df = raw.copy()
df["moisture"] = df["raw_moisture"].rolling(PARAMS["debounce_window"], min_periods=1).median()

start = pd.Timestamp("2025-01-08")
window = df[(df.ts >= start) & (df.ts < start + pd.Timedelta(days=3))]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 6.5), sharex=True, gridspec_kw={"height_ratios": [3, 1.3]})
ax1.plot(window.ts, window.moisture, color="tab:blue", lw=1.3, label="soil moisture (15-min median)")
ax1.axhline(PARAMS["turn_on_threshold"], color="gray", ls="--", lw=1, label=f"ON threshold ({PARAMS['turn_on_threshold']:.0f}%)")
ax1.fill_between(window.ts, 0, 1, where=window.pump, transform=ax1.get_xaxis_transform(),
                 color="tab:orange", alpha=0.35, label="pump ON")
ax1.fill_between(window.ts, 0, 1, where=window.rain, transform=ax1.get_xaxis_transform(),
                 color="tab:cyan", alpha=0.15, label="rain")
ax1.set_ylabel("moisture (%)"); ax1.legend(loc="upper right", fontsize=8, ncol=2)
ax1.set_title("Three days of simulated data")

ax2.plot(window.ts, window.water_level, color="tab:green", lw=1.3)
ax2.axhline(PARAMS["minimum_water_level"], color="gray", ls="--", lw=1)
ax2.set_ylabel("water level"); ax2.set_xlabel("time")
plt.tight_layout(); plt.show()

## B3. Transformasi dari nilai numerik ke category, kemudian ke event

Tahap ini mensimulasikan skenario di mana data sensor per menit dikonversi menjadi category dan hanya perubahan category yang disimpan.

1. **Categorization.** `soil` dibentuk dari kelembaban yang telah dihaluskan, `water` dari tinggi air, dan `time` dari jam.
2. **Compression.** Row baru dibuat hanya ketika salah satu category (termasuk target `pump`) berubah.
3. **Duration.** Lama setiap event disimpan dalam menit dan digunakan sebagai **sample weight**, sehingga state yang berlangsung 8 jam tidak diperlakukan setara dengan state yang berlangsung 3 menit.

Duration **tidak** digunakan sebagai feature. Duration baru diketahui setelah suatu event selesai, sehingga penggunaannya untuk prediksi merupakan information leakage.

In [ ]:
df["soil"] = pd.cut(df["moisture"], [-np.inf, PARAMS["dry_threshold"], PARAMS["wet_threshold"], np.inf],
                    labels=["dry", "moist", "wet"]).astype(str)
df["water"]  = np.where(df["water_level"] < PARAMS["minimum_water_level"], "low", "enough")
df["time"]   = pd.cut(df["ts"].dt.hour, [-1, 5, 11, 17, 23],
                      labels=["night", "morning", "noon", "afternoon"]).astype(str)
df["rain"]   = df["rain"].map({True: "yes", False: "no"})
df["pump"]   = df["pump"].map({True: "ON", False: "OFF"})

FEATURES = ["soil", "water", "rain", "time"]
TARGET = "pump"
category_cols = FEATURES + [TARGET]

changed = (df[category_cols] != df[category_cols].shift()).any(axis=1)
events = df.loc[changed, ["ts"] + category_cols].copy()
events["duration_min"] = (events["ts"].shift(-1) - events["ts"]).dt.total_seconds() / 60
events = events.dropna(subset=["duration_min"]).reset_index(drop=True)   # the last event has no end

print(f"Before compression : {len(df):>7,} rows")
print(f"After compression  : {len(events):>7,} events  (ratio {len(df)/len(events):.1f}x)\n")
events.head(12)

In [ ]:
summary = events.groupby(TARGET).agg(events=("duration_min", "size"),
                                     total_minutes=("duration_min", "sum"),
                                     median_minutes=("duration_min", "median"))
summary["time_share_%"] = (100 * summary["total_minutes"] / summary["total_minutes"].sum()).round(1)
summary["event_share_%"] = (100 * summary["events"] / summary["events"].sum()).round(1)
print("Target classes, by number of events versus time spent:")
summary

In [ ]:
print("Share of minutes per category (%):")
for feature in FEATURES + [TARGET]:
    shares = (100 * df[feature].value_counts(normalize=True)).round(1)
    print(f"  {feature:<6}: " + ", ".join(f"{k} {v}" for k, v in shares.items()))

Output pertama menunjukkan bahwa seluruh category terwakili: `soil = dry` sekitar separuh waktu, `water = low` sekitar sepersepuluh, `rain = yes` sekitar 8%, sedangkan `soil = wet` jarang (sekitar 3%).

Selisih antara `event_share_%` dan `time_share_%` pada tabel sebelumnya juga perlu diperhatikan: pompa ON menempati sekitar seperempat total waktu (24,4%) tetapi hampir separuh event (47,4%), dengan median durasi 9 menit berbanding 67 menit untuk OFF. State ON bersifat pendek dan sering, sedangkan state OFF jarang tetapi panjang. Konsekuensinya, pilihan weighting (dengan atau tanpa duration) mengubah apa yang dianggap penting oleh tree, dan accuracy pada data imbalanced mudah menyesatkan. Evaluation berikutnya karena itu menggunakan balanced accuracy.

## B4. Split temporal dan training

Split acak berpotensi menyebabkan information leakage, karena event yang berurutan sangat mirip akibat autokorelasi temporal. Oleh karena itu digunakan **70% event pertama sebagai training set dan 30% event terakhir sebagai test set**.

Feature categorical dikonversi ke representasi binary (*one-hot encoding*) karena scikit-learn memerlukan input numerik.

In [ ]:
X = pd.get_dummies(events[FEATURES]).astype(int)
y = events[TARGET]
w = events["duration_min"]

k = int(len(events) * 0.7)
X_train, X_test = X.iloc[:k], X.iloc[k:]
y_train, y_test = y.iloc[:k], y.iloc[k:]
w_train, w_test = w.iloc[:k], w.iloc[k:]

print(f"Train: {len(X_train)} events, {events.ts.iloc[0]:%d %b} to {events.ts.iloc[k-1]:%d %b}")
print(f"Test : {len(X_test)} events, {events.ts.iloc[k]:%d %b} to {events.ts.iloc[-1]:%d %b}")
print(f"ON share (by time) -> train {w_train[y_train=='ON'].sum()/w_train.sum():.1%}, test {w_test[y_test=='ON'].sum()/w_test.sum():.1%}")

In [ ]:
tree = DecisionTreeClassifier(criterion="entropy", max_depth=4, random_state=0)   # initial tree; the depth is revisited in B7
tree.fit(X_train, y_train, sample_weight=w_train)

feature_names = list(X.columns)
print(f"Initial tree -> depth: {tree.get_depth()}, leaves: {tree.get_n_leaves()}")

## B5. Interpretasi tree sebagai rule

Setiap leaf diterjemahkan menjadi rule IF-THEN dan diurutkan berdasarkan total waktu yang dicakup leaf tersebut. Condition berulang pada satu feature diringkas (misalnya "bukan noon DAN bukan morning" menjadi `time ∈ {night, afternoon}`). Nilai `correct` adalah proportion waktu pada leaf yang class-nya sesuai dengan prediction (leaf purity).

**Pembacaan visualisasi tree.** Node internal memuat satu condition (misalnya `soil = dry?`) dengan label branch `no`/`yes`. Node leaf memuat class prediction dan share data pada leaf tersebut (di-weight duration). Warna mengikuti class, dan intensitas warna yang lebih pekat menandakan leaf yang lebih pure.

Tree berikut adalah **initial tree** dengan `max_depth=4`, yang ditetapkan sebelum depth optimal ditentukan. Pemilihan depth dilakukan pada B7.

In [ ]:
def print_rules(clf, feature_names):
    possible = {}                                 # feature -> list of possible values
    for name in feature_names:
        feature, value = name.split("_", 1); possible.setdefault(feature, []).append(value)

    t = clf.tree_
    leaves = []
    def walk(node, equals, excludes):
        if t.children_left[node] == t.children_right[node]:          # leaf
            p = t.value[node][0]; p = p / p.sum()
            conditions = []
            for feature in possible:
                if feature in equals:
                    conditions.append(f"{feature} = {equals[feature]}")
                elif excludes.get(feature):
                    remaining = [v for v in possible[feature] if v not in excludes[feature]]
                    conditions.append(f"{feature} = {remaining[0]}" if len(remaining) == 1 else f"{feature} in {{{', '.join(remaining)}}}")
            leaves.append((t.weighted_n_node_samples[node], conditions,
                           clf.classes_[p.argmax()], p.max(), int(t.n_node_samples[node])))
            return
        feature, value = feature_names[t.feature[node]].split("_", 1)
        excluded_next = {key: list(values) for key, values in excludes.items()}; excluded_next.setdefault(feature, []).append(value)
        walk(t.children_left[node], dict(equals), excluded_next)                  # answer No
        walk(t.children_right[node], {**equals, feature: value}, {key: list(values) for key, values in excludes.items()})   # answer Yes
    walk(0, {}, {})

    for weight, conditions, class_label, purity, n in sorted(leaves, key=lambda d: -d[0]):
        condition_text = " AND ".join(conditions) if conditions else "(all conditions)"
        print(f"IF {condition_text}")
        print(f"   THEN pump = {class_label:<3} | {n:>4} events, {weight:>9,.0f} minutes, correct {purity:.0%}\n")

print_rules(tree, feature_names)

In [ ]:
plot_clean_tree(tree, feature_names, "Initial tree (max_depth=4)")

**Interpretasi initial tree (pada run bawaan).** Satu-satunya leaf ON memuat keempat syarat sekaligus: `soil = dry`, `water = enough`, `rain = no`, dan `time` bukan `noon`, dengan purity 98%. Kondisi ini identik dengan rule controller pada B2, sehingga tree merekonstruksi rule ON secara utuh. Seluruh leaf lainnya OFF, dan masing-masing menjawab pertanyaan "mengapa pompa tidak menyala": soil sudah `moist` atau `wet`, jam larangan (`noon`), tangki `low`, atau `rain = yes`.

## B6. Evaluation initial tree pada test set

Tiga metric digunakan, seluruhnya **di-weight duration** agar merepresentasikan waktu nyata dan bukan jumlah event:
- **Accuracy**: proportion waktu di mana prediction benar.
- **Balanced accuracy**: rata-rata *recall* class ON dan OFF. Metric ini tidak bias terhadap class OFF yang dominan.
- **Precision dan recall class ON**: precision adalah proporsi waktu yang diprediksi ON dan benar-benar ON; recall adalah proporsi waktu ON aktual yang berhasil terdeteksi.

Pembanding yang digunakan adalah **majority baseline**, yaitu prediktor yang selalu memilih class dengan durasi terpanjang pada training set (OFF). Tree baru dianggap bermakna jika mengungguli baseline ini. Baseline sudah mencapai accuracy 0,743 hanya karena pompa lebih sering OFF, sedangkan recall class ON-nya 0. Hal ini menjelaskan mengapa accuracy saja kurang informatif pada data yang imbalanced.

In [ ]:
def metrics(y_true, y_pred, weights):
    actual_on = (y_true == "ON"); predicted_on = (y_pred == "ON")
    tp = weights[actual_on & predicted_on].sum(); fp = weights[~actual_on & predicted_on].sum(); fn = weights[actual_on & ~predicted_on].sum()
    return {
        "accuracy": np.average(y_true == y_pred, weights=weights),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred, sample_weight=weights),
        "precision_ON": tp / (tp + fp) if (tp + fp) else np.nan,
        "recall_ON": tp / (tp + fn) if (tp + fn) else np.nan,
    }

def confusion_matrix(y_true, y_pred, weights):
    cm = (pd.DataFrame({"actual": y_true, "predicted": y_pred, "minutes": weights})
            .pivot_table(index="actual", columns="predicted", values="minutes", aggfunc="sum", fill_value=0)
            .reindex(index=["OFF", "ON"], columns=["OFF", "ON"], fill_value=0))
    cm.index = ["Actual OFF", "Actual ON"]; cm.columns = ["Predicted OFF", "Predicted ON"]
    return cm.round(0).astype(int)

majority_class = y_train.groupby(y_train).apply(lambda s: w_train[s.index].sum()).idxmax()
pred_baseline = pd.Series(majority_class, index=y_test.index)
pred_test = pd.Series(tree.predict(X_test), index=y_test.index)

results = pd.DataFrame({
    f"baseline (always {majority_class}), test": metrics(y_test, pred_baseline, w_test),
    "initial tree, train": metrics(y_train, pd.Series(tree.predict(X_train), index=y_train.index), w_train),
    "initial tree, test": metrics(y_test, pred_test, w_test),
}).T.round(3)
results

In [ ]:
print("Confusion matrix of the initial tree on the test data, in MINUTES (not event counts):")
confusion_matrix(y_test, pred_test, w_test)

## B7. Pruning: pemilihan depth

Tree yang lebih dalam berpotensi memodelkan noise (overfitting). Beberapa value `max_depth` dibandingkan, dan dipilih tree **paling sederhana dengan performa dalam toleransi 0,01 dari balanced accuracy terbaik** pada test set.

Catatan metodologis: pemilihan depth berdasarkan test set, kemudian pelaporan score pada test set yang sama, menimbulkan optimistic bias. Untuk studi yang lebih ketat disarankan penggunaan cross-validation berurutan (`TimeSeriesSplit`) atau validation set yang terpisah.

In [ ]:
rows = []
for depth in [1, 2, 3, 4, 5, 6, 8, None]:
    clf = DecisionTreeClassifier(criterion="entropy", max_depth=depth, random_state=0).fit(X_train, y_train, sample_weight=w_train)
    rows.append({
        "max_depth": "no limit" if depth is None else depth,
        "leaves": clf.get_n_leaves(),
        "train_balanced_accuracy": balanced_accuracy_score(y_train, clf.predict(X_train), sample_weight=w_train),
        "test_balanced_accuracy": balanced_accuracy_score(y_test, clf.predict(X_test), sample_weight=w_test),
    })
depth_table = pd.DataFrame(rows)

best_test = depth_table["test_balanced_accuracy"].max()
selected = depth_table[depth_table["test_balanced_accuracy"] >= best_test - 0.01].iloc[0]
depth_table["chosen"] = np.where(depth_table["max_depth"].astype(str) == str(selected["max_depth"]), "<--", "")
display(depth_table.round(3))
print(f"Simplest tree within 0.01 of the best: max_depth = {selected['max_depth']} ({int(selected['leaves'])} leaves)")

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 3.8))
ax.plot(depth_table["leaves"], depth_table["train_balanced_accuracy"], "o-", label="train")
ax.plot(depth_table["leaves"], depth_table["test_balanced_accuracy"], "s-", label="test")
ax.set_xlabel("number of leaves"); ax.set_ylabel("balanced accuracy")
ax.set_title("Tree complexity vs performance"); ax.legend()
plt.tight_layout(); plt.show()

**Interpretasi (pada run bawaan).**
- Depth 1 tidak lebih baik daripada baseline (balanced accuracy 0,5): satu pertanyaan tidak mampu memisahkan ON dari OFF, karena pompa ON memerlukan *kombinasi* beberapa kondisi.
- Depth 2 menangkap sebagian besar struktur (test 0,883), dan depth 3 menangkap kombinasi `soil = dry`, `water = enough`, dan `time` bukan `noon` (test 0,953).
- Depth 4 hanya menambah sekitar 0,006 balanced accuracy (0,959), di bawah toleransi 0,01. Split tambahan yang paling relevan adalah `rain` pada branch ON, yang terlihat pada B8 sebagai kenaikan precision ON.
- Dari depth 4 sampai tanpa batas, skor tidak berubah (training 0,943; test 0,959), padahal jumlah leaf naik dari 12 menjadi 25 dan berhenti di sana. Leaf tambahan tidak memberi manfaat. Plateau ini mencerminkan ruang feature yang kecil (empat feature categorical, paling banyak 3 x 2 x 2 x 4 = 48 kombinasi) dan data yang hampir deterministik, bukan bukti bahwa tree robust terhadap overfitting pada data nyata yang lebih berisik.
- Aturan toleransi 0,01 adalah pilihan, bukan hukum. Jika biaya menyiram saat hujan dianggap tinggi, depth 4 lebih sesuai.

## B8. Final tree: rule, performa, dan feature yang digunakan

Tree di-training ulang menggunakan depth yang terpilih pada B7.

In [ ]:
final_depth = None if str(selected["max_depth"]) == "no limit" else int(selected["max_depth"])
tree_final = DecisionTreeClassifier(criterion="entropy", max_depth=final_depth, random_state=0)
tree_final.fit(X_train, y_train, sample_weight=w_train)

print(f"Final tree: max_depth = {final_depth}, {tree_final.get_n_leaves()} leaves\n")
print_rules(tree_final, feature_names)

In [ ]:
plot_clean_tree(tree_final, feature_names, f"Final tree (max_depth={final_depth})")

**Cara membaca final tree (pada run bawaan).** Setiap leaf adalah satu rule IF-THEN. Hanya satu leaf yang memprediksi ON; enam leaf lainnya OFF dan masing-masing mewakili alasan mengapa pompa tidak menyala.

| Leaf | Kondisi | Prediction | Purity | Alasan menurut rule simulator |
|---|---|---|---|---|
| 1 | `soil = dry`, `water = enough`, `time` bukan `noon` | **ON** | 92% | semua syarat ON terpenuhi |
| 2 | `soil = dry`, `water = low`, `time` bukan `noon` | OFF | 97% | tangki rendah |
| 3 | `soil = dry`, `rain = no`, `time = noon` | OFF | 98% | jam larangan |
| 4 | `soil = dry`, `rain = yes`, `time = noon` | OFF | 100% | jam larangan (sekaligus hujan) |
| 5 | `soil = moist`, `rain = no` | OFF | 95% | soil di atas threshold |
| 6 | `soil = moist`, `rain = yes` | OFF | 99% | soil di atas threshold |
| 7 | `soil = wet` | OFF | 100% | soil di atas threshold |

Leaf 3 dan 4 berbagi prediction dan alasan yang sama, begitu pula leaf 5, 6, dan 7. Split yang memisahkan mereka tidak mengubah keputusan; split itu dibuat karena criterion entropy mengejar purity leaf, bukan perubahan prediction. Untuk keputusan, tree ini efektif satu kalimat: **ON jika tanah dry, air cukup, dan bukan jam siang**. `rain` tidak muncul sebagai alasan OFF karena split `rain` pada branch ON baru terjadi pada depth 4.

In [ ]:
pred_final = pd.Series(tree_final.predict(X_test), index=y_test.index)
comparison = pd.DataFrame({
    f"baseline (always {majority_class})": metrics(y_test, pred_baseline, w_test),
    "initial tree (depth 4)": metrics(y_test, pred_test, w_test),
    f"final tree (depth {final_depth})": metrics(y_test, pred_final, w_test),
}).T.round(3)
print("All on TEST data, weighted by duration:")
display(comparison)
print("\nConfusion matrix of the final tree (minutes):")
confusion_matrix(y_test, pred_final, w_test)

**Interpretasi (pada run bawaan).** Final tree (depth 3) dan initial tree (depth 4) memiliki recall ON hampir sama (0,921 dan 0,917), tetapi precision ON berbeda: 0,957 dibanding 1,000. Selisih ini berasal dari split `rain` yang hanya ada pada depth 4. Pada depth 3, kondisi `dry`, `enough`, dan jam yang diizinkan selalu diprediksi ON, termasuk saat hujan, padahal controller tidak menyalakan pompa saat hujan.

Penambahan depth tidak memperbaiki recall. Menit ON yang terlewat (missed ON) tidak dapat dipisahkan oleh feature yang tersedia, karena berasal dari manual override dan dari periode ON yang masih berlangsung setelah `soil` berpindah ke `moist` (efek `min_on_minutes` dan chattering di sekitar threshold). Dengan kata lain, recall dibatasi oleh label noise, bukan oleh kapasitas tree.

Pemilihan model bergantung pada biaya relatif kedua jenis kesalahan: memprediksi ON padahal OFF (false alarm) dibandingkan melewatkan ON. Metric evaluasi sebaiknya ditetapkan sebelum hasil diobservasi.

### Feature importance

Feature importance dijumlahkan per feature asli, bukan per column one-hot, dan mengukur kontribusi terhadap penurunan entropy pada training set, bukan pengaruh kausal terhadap pompa.

Pada run bawaan, `time` menempati urutan teratas (0,424), disusul `soil` (0,333) dan `water` (0,239). `time` unggul karena jam larangan memotong massa waktu yang besar dari kondisi `dry` (leaf `dry`, `noon` mencakup sekitar 16.000 menit pada training set). `rain` hampir nol (0,004) karena rule hujan hanya berlaku pada irisan kecil data, yaitu kondisi `dry`, `enough`, jam diizinkan, dan hujan, sehingga split `rain` pada depth 3 hanya terjadi pada leaf bermassa kecil.

In [ ]:
importances = pd.Series(tree_final.feature_importances_, index=feature_names)
importances_by_feature = importances.groupby(lambda c: c.split("_", 1)[0]).sum().reindex(FEATURES).sort_values()

fig, ax = plt.subplots(figsize=(7, 3.2))
importances_by_feature.plot.barh(ax=ax, color="tab:blue")
ax.set_xlabel("share of entropy decrease (total = 1)")
ax.set_title("Feature importance, final tree")
for i, v in enumerate(importances_by_feature.values):
    ax.text(v + 0.005, i, f"{v:.3f}", va="center", fontsize=9)
plt.tight_layout(); plt.show()

## B9. Prediction pada skenario baru

Beberapa skenario hipotetis digunakan untuk memeriksa perilaku tree sebagai decision maker. Kolom `true_rule` adalah keputusan menurut rule simulator, yang berfungsi sebagai sanity check. Row yang menunjukkan perbedaan antara prediction tree dan `true_rule` paling informatif untuk dianalisis.

In [ ]:
scenarios = pd.DataFrame([
    ("dry",   "enough", "no",  "night"),
    ("dry",   "enough", "no",  "afternoon"),
    ("dry",   "enough", "no",  "noon"),
    ("dry",   "low",    "no",  "afternoon"),
    ("dry",   "enough", "yes", "afternoon"),
    ("moist", "enough", "no",  "morning"),
    ("wet",   "enough", "no",  "night"),
], columns=FEATURES)

def one_hot_encode(cat_df):
    encoded = pd.DataFrame(0, index=cat_df.index, columns=X.columns)
    for feature in FEATURES:
        for i, value in cat_df[feature].items():
            col = f"{feature}_{value}"
            if col in encoded.columns: encoded.loc[i, col] = 1
    return encoded

X_scenarios = one_hot_encode(scenarios)
scenarios["tree_prediction"] = tree_final.predict(X_scenarios)
scenarios["proba_ON"] = tree_final.predict_proba(X_scenarios)[:, list(tree_final.classes_).index("ON")].round(2)
scenarios["true_rule"] = np.where((scenarios.soil == "dry") & (scenarios.water == "enough")
                                  & (scenarios.rain == "no") & (scenarios.time != "noon"), "ON", "OFF")
scenarios

**Analisis selisih.** Baris dengan `rain = yes` (dry, enough, rain, afternoon) adalah satu-satunya skenario yang diprediksi berbeda dari `true_rule`: tree memprediksi ON dengan `proba_ON` 0,92, padahal controller tidak menyalakan pompa saat hujan.

Penyebabnya konsisten dengan B7 dan B8: pada depth 3 branch ON tidak memuat split `rain`, karena irisan data yang terdampak kecil. Pada initial tree (depth 4), leaf `dry`, `enough`, `rain = yes` hanya mencakup sekitar 1.300 menit, dibandingkan sekitar 18.700 menit pada leaf ON. Tree hanya mempelajari pola yang cukup besar massanya untuk menurunkan entropy.

Perlu dicatat bahwa `proba_ON` 0,92 bukan probability terkalibrasi. Nilai itu adalah proporsi waktu ON (di-weight duration) pada leaf training, sehingga setiap skenario yang jatuh pada leaf yang sama mendapat nilai yang sama.

Perilaku yang tepat bergantung pada data hasil simulasi (di sini `SEED = 42`).

In [ ]:
top_features = importances_by_feature.sort_values(ascending=False)
final_metrics = metrics(y_test, pred_final, w_test)
print("AUTOMATIC SUMMARY")
print("-----------------")
print(f"- Data    : {len(df):,} minute rows -> {len(events)} events ({len(df)/len(events):.0f}x more compact).")
print(f"- Tree    : max_depth {final_depth}, {tree_final.get_n_leaves()} leaves (selected in B7).")
print(f"- Features: most influential = {top_features.index[0]} ({top_features.iloc[0]:.2f}), "
      f"then {top_features.index[1]} ({top_features.iloc[1]:.2f}); least = {top_features.index[-1]} ({top_features.iloc[-1]:.2f}).")
print(f"- Test    : balanced accuracy {final_metrics['balanced_accuracy']:.1%} vs baseline {comparison.iloc[0]['balanced_accuracy']:.1%}.")
print(f"- ON class: precision {final_metrics['precision_ON']:.1%}, recall {final_metrics['recall_ON']:.1%} (duration-weighted).")

---
# Bagian C. Keterbatasan dan Pengembangan Lanjutan

## Keterbatasan (threats to validity)

1. **Sirkularitas.** Target dibangkitkan oleh rule yang ditulis penulis, sehingga "keberhasilan" tree sebagian besar berarti ia merekonstruksi rule tersebut. Pada sistem nyata, rule controller belum tentu diketahui, dan pada kondisi itulah tree bermanfaat sebagai alat penjelas (explanatory tool).
2. **Keselarasan controller dan feature.** Controller membaca filtered reading yang sama dengan feature `soil`, dan batas `dry` sama dengan threshold pompa. Pada sistem nyata, sinyal pengambil keputusan dan feature yang tersedia umumnya tidak identik (sensor, filter, dan threshold berbeda), sehingga mismatch tambahan akan menurunkan performa tree.
3. **Parameter ilustratif.** Tanpa estimasi dari data lapangan, parameter seperti laju evaporasi tidak dapat diklaim merepresentasikan sawah tertentu.
4. **Hysteresis dan memory.** Controller nyata sering bergantung pada riwayat (misalnya pompa tetap ON hingga kelembaban mencapai threshold atas). Feature snapshot seperti yang digunakan di sini tidak menangkap riwayat. Trend feature atau status pompa sebelumnya dapat membantu, dengan risiko menjadi proxy dari target itu sendiri (trend kelembaban adalah *akibat* pompa, bukan penyebabnya).
5. **Threshold category yang arbitrer.** Batas `dry`/`moist`/`wet` ditetapkan di awal. Jika batas dipilih dengan mengamati target, hasil menjadi bias.
6. **Satu split temporal.** Evaluation hanya menggunakan satu test split. Variasi antar seed belum diukur, dan depth dipilih berdasarkan test set yang sama (optimistic bias).
7. **Duration weighting adalah keputusan desain.** Tree dapat berbeda dengan dan tanpa weight, dan tidak ada pilihan yang sepenuhnya netral.
8. **Single decision tree.** Memadai untuk tujuan interpretasi, namun pada data nyata umumnya kurang stabil dibandingkan random forest atau gradient boosting.

## Pengembangan lanjutan

1. Analisis sensitivitas terhadap `SEED`: apakah feature dominan dan depth terpilih stabil antar seed.
2. Pengaruh `sensor_noise` yang lebih tinggi (misalnya 1,5) terhadap performa pada test set dan jumlah event setelah compression.
3. Pengaruh `sensor_lag = 0` dan `min_on_minutes = 1` terhadap jumlah event dan proporsi menit ON yang jatuh pada category `dry`.
4. Replikasi mismatch category: setel `dry_threshold = 42` (di atas threshold pompa) dan amati penurunan purity leaf ON.
5. Ablation tanpa `sample_weight`: perbandingan rule yang dihasilkan.
6. Pengaturan `forbidden_hours = (0, 0)` untuk menguji apakah `time` tereliminasi dari tree.
7. Perbandingan `criterion="entropy"` dan `"gini"`: seberapa sering struktur tree berbeda.
8. Kalibrasi `PARAMS` dari data sensor nyata, kemudian perbandingan proportion waktu pompa ON dan distribusi duration antara data nyata dan sintetis sebagai validasi fidelity simulator.